## Tracking of a Fish-position dataset optimised over a parameter grid, using likelihoods

In [ ]:
## Non-optimised params
num_timesteps= 500
number_particles = 20
seed = 1 # Random seem for reproducibility
c=10

In [ ]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import pandas as pd
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import CovarianceMatrices, StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater

# Step 1: Load the CSV file- With thanks to Chloe Chung for the data!
excel_folder = fr"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TrackedDatasets\FishData\train"
files = [
    rf"ZebraFish-01",
    rf"ZebraFish-02",
    rf"ZebraFish-03",
    rf"ZebraFish-04",
    ]

start_time=datetime.now()

trajectory_data={}
all_fish_id=[]
for txt_file in files:
    file_path = excel_folder + "\\" + txt_file + "\gt\gt.txt"

    # Each line has 19 columns, but we only want these:
    col_names = ["frame", "id", "3d_x", "3d_y", "3d_z",
                 "camT_x","camT_y","camT_left","camT_top",
                 "camT_width","camT_height","camT_occlusion",
                 "camF_x","camF_y","camF_left","camF_top",
                 "camF_width","camF_height","camF_occlusion"]
    
    # Read the txt as CSV
    df = pd.read_csv(file_path, sep=",", header=None, names=col_names)
    df_needed = df[["frame","id","3d_x","3d_y","3d_z"]]
    grouped = df_needed.groupby("id")

    if txt_file not in trajectory_data:
        trajectory_data[txt_file] = {}
    
    for fish_id, group_df in grouped:
        frames = group_df["frame"].to_numpy()
        x_arr  = group_df["3d_x"].to_numpy()
        y_arr  = group_df["3d_y"].to_numpy()
        z_arr  = group_df["3d_z"].to_numpy()
        print(fish_id,txt_file)
        trajectory_data[txt_file][fish_id] = {}

        timesteps = [start_time + timedelta(milliseconds=int(frame_idx)) for frame_idx in frames]
        timesteps = timesteps[:num_timesteps]

        # Build separate Tracks of Detections for position vs velocity
        measurement_track = Track()
        plottable_track= Track()
        for i in range(num_timesteps):
            timestamp = timesteps[i]
            meas = np.array([x_arr[i], y_arr[i], z_arr[i]])
            metadata={"file": txt_file, "fish_id": fish_id}
            detection = Detection(
                state_vector=StateVector(meas),
                timestamp=timestamp,
                metadata=metadata)        
            plottable_state = GroundTruthState(
                state_vector=StateVector(meas),
                timestamp=timestamp,
                metadata=metadata)
            measurement_track.append(detection)
            plottable_track.append(plottable_state)
        x_std = np.std(x_arr)
        y_std = np.std(y_arr)
        z_std = np.std(z_arr)
        prior_mean = [x_arr[0], y_arr[0], z_arr[0]]
        prior_covar = np.diag([x_std**2, y_std**2, z_std**2])
        # Store in the dictionary
        trajectory_data[txt_file][fish_id] = {
                "measurements"     : measurement_track,
                "plottable_tracks" : plottable_track,
                "prior_stats"      : (prior_mean, prior_covar)
            }

In [ ]:
#Determine assumed measurement std.
meas_sigma=0.00005
meas_sig2 = meas_sigma**2

In [ ]:
## Initiate Priors
for txt_file, file_data in trajectory_data.items():
    # file_data is the dictionary for that file, keyed by fish_id
    for fish_id, fish_dict in file_data.items():
        prior_mean,prior_covar=trajectory_data[txt_file][fish_id]['prior_stats']
        # Sample from the prior Gaussian distribution
        states = multivariate_normal.rvs(
            mean=prior_mean,
            cov=prior_covar,  # Covariance for the initial state
            size=number_particles
        )

        # Define covariance for particle array
        covars = [prior_covar for _ in range(number_particles)]

        # Create prior particle state
        lp_prior = MarginalisedParticleState(
            state_vector=StateVectors(states.T),  # Transpose states to shape (2, N)
            covariance=CovarianceMatrices(covars).T,  # Covariance matrix
            weight=np.array([Probability(1 / number_particles)] * number_particles),
            timestamp=start_time-timedelta(milliseconds=1)
        )
        gp_prior=GaussianState(state_vector=prior_mean,
                                covar=prior_covar,
                                timestamp=start_time)
        # Store them in the dictionary
        trajectory_data[txt_file][fish_id]['prior_states']=(lp_prior,gp_prior) # lp and gp prior objects

In [ ]:
## Outlining the parameter grids over which we'll estimate
# RandomWalk parameters    
#  
#q or sigma_W, std of noise
noise_diff_coeffs = np.logspace(-12, -8, num= 3)
#alpha param, how heavy-tailed levy distr is. avoid alpha=1      
a,b=2,2
pre_1_alpha,post_1_alpha, alpha_values= np.linspace(0.5, 0.9, a), np.linspace(1.1,1.9,b), np.zeros(a+b)
alpha_values[:a],alpha_values[a:a+b]=pre_1_alpha,post_1_alpha
# See below, mu_w=0 held const.

#Levy Langevin or Ulhenbeck Orstein params:
damping_coeffs= np.linspace(0.01, 0.5, 3)

In [ ]:
## Generates the necessary predictor dicts for all the components to save time in the loop
from stonesoup.models.transition.levy_linear import LevyRandomWalk
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, OrnsteinUhlenbeck

resampler = SystematicResampler()

lp_predictors= {}
gp_predictors = {}

for txt_file, file_data in trajectory_data.items():
    lp_predictors[txt_file]={}
    gp_predictors[txt_file]={}
    for fish_id, fish_dict in file_data.items():
        lp_predictors[txt_file][fish_id]= {}
        gp_predictors[txt_file][fish_id]= {}
        for q in noise_diff_coeffs: 
            for theta in damping_coeffs: # how quickly decay back to zero occurs (higher=quicker)
                for alpha in alpha_values: # how heavy-tailed distr is (lower=heavier)
                        
                        #Generates all the Levy process predictors and updaters
                        #driver has sigma_W=1, so std. controlled by noise_diff_coeff input into model
                        lp_driver = AlphaStableNSMDriver(mu_W=0, sigma_W2=q**2, seed=seed, c=c, alpha=alpha, noise_case=NoiseCase(2))                            
                        levy_rw_x=LevyRandomWalk(driver=lp_driver,noise_diff_coeff=q)
                        levy_rw_y=levy_rw_x
                        levy_rw_z=levy_rw_x
                        lp_transition_model=CombinedLinearLevyTransitionModel([levy_rw_x,levy_rw_y,levy_rw_z])
                        lp_predictor = MarginalisedParticlePredictor(transition_model=lp_transition_model)
                        lp_predictors[txt_file][fish_id][(q,theta, alpha)] = lp_predictor

                #Generates all the Gaussian process predictors and updaters
                gp_RW_x= RandomWalk(noise_diff_coeff=q)
                gp_RW_y=gp_RW_x
                gp_RW_z=gp_RW_x
                gp_transition_model= CombinedLinearGaussianTransitionModel([gp_RW_x,gp_RW_y,gp_RW_z])
                gp_predictor = KalmanPredictor(gp_transition_model)
                gp_predictors[txt_file][fish_id][(q,theta)] = gp_predictor     

In [ ]:
## Build the updaters that depend on measurement model only
from stonesoup.updater.tests.conftest import measurement_model

measurement_model = LinearGaussian(
ndim_state=3,  # State vector dimensions: [price, dP/dt] for the models
mapping=[0,1,2],  # Map the measurement to the 'price' dimension
noise_covar=np.diag([meas_sig2]*3))

lp_updater = MarginalisedParticleUpdater(measurement_model, resampler)
gp_updater = KalmanUpdater(measurement_model)   

In [ ]:
## Filtering and likelihood calculation
from scipy.special import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

#Track dicts
lp_tracks = {}
gp_tracks = {}

#Likelihood dicts
lp_likelihoods = {}
gp_likelihoods = {}



for txt_file, file_data in trajectory_data.items():
    lp_likelihoods[txt_file]={}
    gp_likelihoods[txt_file]={}
    lp_tracks[txt_file]={}
    gp_tracks[txt_file]={}
    for fish_id, fish_dict in file_data.items():
        measurements=trajectory_data[txt_file][fish_id]['measurements']
        lp_likelihoods[txt_file][fish_id] = {}
        gp_likelihoods[txt_file][fish_id] = {}
        lp_tracks[txt_file][fish_id]={}
        gp_tracks[txt_file][fish_id]={}

        for i, meas in enumerate(measurements):
            # Possibly you'd also keep track of prior states for each param combination
            # For each alpha, etc.:
            for q in noise_diff_coeffs:
                for theta in damping_coeffs:
                    for alpha in alpha_values:
                        if (q,theta,alpha) not in lp_likelihoods[txt_file][fish_id]:
                            lp_tracks[txt_file][fish_id][(q,theta, alpha)]=Track()
                            prior=trajectory_data[txt_file][fish_id]['prior_states'][0]
                            lp_likelihoods[txt_file][fish_id][(q,theta, alpha)] = 0.0
                        else:
                            lp_tracks[txt_file][fish_id][(q,theta, alpha)]
                            prior = lp_tracks[txt_file][fish_id][(q,theta, alpha)][-1]
                            
                        lp_predictor = lp_predictors[txt_file][fish_id][(q,theta, alpha)]
                        lp_prediction = lp_predictor.predict(prior, timestamp=meas.timestamp)
                        lp_hypothesis = SingleHypothesis(lp_prediction, meas)                    
                        lp_post = lp_updater.update(lp_hypothesis)
                        lp_tracks[txt_file][fish_id][(q,theta, alpha)].append(lp_post)
                        # Accumulate log-likelihood
                        lp_likelihoods[txt_file][fish_id][(q,theta, alpha)] += logsumexp(lp_updater.measurement_model.logpdf(meas,lp_post))-np.log(number_particles*num_timesteps)

                    if (q,theta) not in gp_likelihoods[txt_file][fish_id]:
                        gp_tracks[txt_file][fish_id][(q,theta)]=Track()
                        prior=trajectory_data[txt_file][fish_id]['prior_states'][1]
                        gp_likelihoods[txt_file][fish_id][(q,theta)] = 0.0
                    else:
                        gp_tracks[txt_file][fish_id][(q,theta)]
                        prior = gp_tracks[txt_file][fish_id][(q,theta)][-1]

                    gp_predictor = gp_predictors[txt_file][fish_id][(q,theta)]
                    gp_prediction = gp_predictor.predict(prior, timestamp=meas.timestamp)
                    gp_hypothesis = SingleHypothesis(gp_prediction, meas)
                    gp_post = gp_updater.update(gp_hypothesis)
                    gp_tracks[txt_file][fish_id][(q,theta)].append(gp_post)
                    
                    gp_likelihoods[txt_file][fish_id][(q,theta)] += gp_updater.measurement_model.logpdf(meas,gp_post) -np.log(num_timesteps)

            print(f"file {txt_file}, (fish {fish_id}/{len(fish_dict)}, {i+1}/{len(measurements)} measurements)")

In [ ]:
## Define likelihood organiser and output optimal params
def summarize_top_likelihoods(txt_file,fish_id,lp_likelihoods,gp_likelihoods, top_n=5):
    """
    Summarize and print the top-N parameter configurations with the highest 
    log-likelihood for both the Lévy process and Gaussian process models.
    
    Parameters
    ----------
    lp_likelihoods : dict
        Dictionary keyed by (mu_W, theta, alpha, q, meas_sig2),
        with values = total (log) likelihood.
    gp_likelihoods : dict
        Dictionary keyed by ((q,theta), meas_sig2), with values = total (log) likelihood.
    top_n : int, optional
        Number of highest-likelihood entries to show for each model. Default=5.
    
    Returns
    -------
    list_of_top_lp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Lévy model.
    list_of_top_gp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Gaussian process model.
    """
    fish_id_lp_likelihoods, fish_id_gp_likelihoods = lp_likelihoods[txt_file][fish_id], gp_likelihoods[txt_file][fish_id]
    # --- 1) Sort the Lévy model likelihoods ---
    #   lp_likelihoods is keyed by (mu_W, theta, alpha, q, meas_sig2)
    #   The value is the total log-likelihood
    # We'll get fish_ids as ((q,theta, alpha), loglike)
    list_of_lp = list(fish_id_lp_likelihoods.items())
    # Sort descending by log-likelihood
    list_of_lp.sort(key=lambda x: x[1], reverse=True)
    # Take top_n
    list_of_top_lp = list_of_lp[:top_n]

    # --- 2) Sort the Gaussian process likelihoods ---
    #   gp_likelihoods is keyed by (q,theta)
    list_of_gp = list(fish_id_gp_likelihoods.items())
    list_of_gp.sort(key=lambda x: x[1], reverse=True)
    list_of_top_gp = list_of_gp[:top_n]

    # --- 3) Print summary in a neat format ---
    print(f"=== Lévy Langevin Process - {txt_file}, Fish {fish_id},  -Top {top_n} Log-Likelihoods ===")
    for rank, ((q,theta, alpha), loglike) in enumerate(list_of_top_lp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | q={q}, alpha={alpha}") #, theta={theta} , mu={mu_W}")

    print("")
    print(f"=== Gaussian Ornstein_Uhlenbeck - {txt_file}, Fish {fish_id}, -Top {top_n} Log-Likelihoods ===")
    for rank, ((q,theta), loglike) in enumerate(list_of_top_gp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | q={q}") #, theta={theta}")

    return list_of_top_lp, list_of_top_gp

In [ ]:
## Optimal parameters used for tracks and plotting
lp_optimal_params ={}
gp_optimal_params ={}
Optimal_lp_tracks= {}
Optimal_gp_tracks={}
for txt_file, file_data in trajectory_data.items():
    lp_optimal_params[txt_file] = {}
    gp_optimal_params[txt_file] ={}
    Optimal_lp_tracks[txt_file]= {}
    Optimal_gp_tracks[txt_file]={}
    for fish_id, fish_dict in file_data.items():
        list_of_top_lp, list_of_top_gp = summarize_top_likelihoods(txt_file,fish_id,lp_likelihoods,gp_likelihoods, top_n=5)
        lp_optimal_params[txt_file][fish_id] = list_of_top_lp[0][0] 
        gp_optimal_params[txt_file][fish_id] = list_of_top_gp[0][0]
        Optimal_lp_tracks[txt_file][fish_id] = lp_tracks[txt_file][fish_id][lp_optimal_params[txt_file][fish_id]]
        Optimal_gp_tracks[txt_file][fish_id] = gp_tracks[txt_file][fish_id][gp_optimal_params[txt_file][fish_id]]

In [ ]:
## Whether to calculate the smoothed trajectories and how to plot them

#1D dimension, 0,1,2 are x,y,z
i=0
save_plots1D =True
show_plots1D=False
uncertainty=True
particle=False
plot_particle_paths=False
plot_smooth=False

#2D mapping, 0,1,2 are x,y,z
mapping_2D=[0,1]
save_plots2D=True
show_plots2D=False
uncertainty2D=True
particle2D=False
plot_particle_paths2D=False
plot_smooth2D=False

#3D
save_plots3D =True
show_plots3D=False

In [ ]:
##Plotting stuff
#  Path to save plots in
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\FishPlots"

colors_dict={
1:'blue',
2:'#00CC96',
3:'#FFA15A',
4:'#B6E880',
5:'#AB63FA'
}

if plot_smooth is True:
    from stonesoup.smoother.particle import CarterKohnSmoother, MarginalisedKalmanSmoother, ParticleSmoother
    particlesmoother=ParticleSmoother()
    RTSsmoother=MarginalisedKalmanSmoother()

In [ ]:
## Plotting
particle_plotter_dict = {}
for txt_file, file_data in trajectory_data.items():
    file_path = Path(folder_path + rf"\1D{txt_file}.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    particle_plotter_dict[txt_file]= Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.ONE,axis_labels=['pos','Time'])
    for fish_id, fish_dict in file_data.items(): 
        LP_track= Optimal_lp_tracks[txt_file][fish_id]
        GP_track= Optimal_gp_tracks[txt_file][fish_id]

        particle_plotter_dict[txt_file].plot_ground_truths(trajectory_data[txt_file][fish_id]['plottable_tracks'], [i],line=dict(width=1, color=colors_dict[fish_id]), 
                                                               truths_label=f"Fish {fish_id} gt")
        particle_plotter_dict[txt_file].plot_tracks(LP_track, [i],mode="lines", 
                                                    uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, 
                                                    line=dict(width=1, color=colors_dict[fish_id]),
                                                    track_label=f"Levy RW, Fish {fish_id}")
        particle_plotter_dict[txt_file].plot_tracks(GP_track, [i],mode="lines",
                                                    uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, 
                                                    line=dict(width=1, color=colors_dict[fish_id], dash='dot'),
                                                    track_label=f"Gaussian RW, Fish {fish_id}")
        if plot_smooth is True:
            culled_track=particlesmoother.particle_paths(track=LP_track)
            RTS_track=RTSsmoother.smooth(track=LP_track)
            particle_plotter_dict[txt_file].plot_tracks(culled_track, [i],mode="lines",
                                                        uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, 
                                                        line=dict(width=1, color=colors_dict[fish_id]),
                                                        track_label=f"culled, Fish {fish_id}")
            particle_plotter_dict[txt_file].plot_tracks(RTS_track, [i],mode="lines",
                                                        uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, 
                                                        line=dict(width=1, color=colors_dict[fish_id]),
                                                        track_label=f"RTS RW, Fish {fish_id}")
    particle_plotter_dict[txt_file].fig.update_layout( 
        plot_bgcolor="white",  # Set background color to white
        xaxis=dict(
            showgrid=True,
            gridcolor="gray",      # Keep the grid
            title=dict(text="Time", font=dict(size=20)),  # Add large fish_id
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="gray",      # Keep the grid
            title=dict(text=fish_id, font=dict(size=20)),  # Add large fish_id
        ),
        legend=dict(
            font=dict(size=15),       # Make the legend font larger
            # orientation='v',
            # xanchor="auto",         # Center the legend
            # yanchor="auto",           # Align the legend to the bottom of the plot
            bordercolor="Black",
            borderwidth=3,
            # y=+0.45,                   # Position it above the graph
            # x=0.6                    # Center it horizontally
        ),
    )
    if save_plots1D:
        particle_plotter_dict[txt_file].fig.write_html(str(file_path))
    if show_plots1D:
        particle_plotter_dict[txt_file].fig.show()

In [ ]:
particle_plotter_dict_2D ={}
for txt_file, file_data in trajectory_data.items():
    particle_plotter_dict_2D[txt_file]={}
    for fish_id, fish_dict in file_data.items():
        particle_plotter_dict_2D[txt_file][fish_id]=Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.TWO)
        file_path = Path(folder_path + rf"\2D{txt_file}\Fish{fish_id}.html")
        file_path.parent.mkdir(parents=True, exist_ok=True)
        LP_track= Optimal_lp_tracks[txt_file][fish_id]
        GP_track= Optimal_gp_tracks[txt_file][fish_id]
        particle_plotter_dict_2D[txt_file][fish_id].plot_ground_truths(trajectory_data[txt_file][fish_id]['plottable_tracks'], mapping_2D, 
                                                        truths_label=f"Fish {fish_id} gt")
        particle_plotter_dict_2D[txt_file][fish_id].plot_tracks(LP_track, mapping_2D,
                                                        mode="lines",line=dict(width=1, color=colors_dict[fish_id]),
                                                        uncertainty=uncertainty2D,particle=particle2D,plot_particle_paths=plot_particle_paths2D,   
                                                        track_label=f"Levy RW, Fish {fish_id}")
        
        particle_plotter_dict_2D[txt_file][fish_id].plot_tracks(GP_track, mapping_2D,
                                                        mode="lines", line=dict(width=1, color=colors_dict[fish_id], dash='dot'),
                                                        uncertainty=uncertainty2D,particle=particle2D,plot_particle_paths=plot_particle_paths2D,   
                                                        track_label=f"Gaussian RW, Fish {fish_id}")
        if plot_smooth is True:
            culled_track=particlesmoother.particle_paths(track=LP_track)
            RTS_track=RTSsmoother.smooth(track=LP_track)
            particle_plotter_dict[txt_file][fish_id].plot_tracks(culled_track, mapping_2D,mode="lines",
                                                        uncertainty=uncertainty2D,particle=particle2D,plot_particle_paths=plot_particle_paths2D, 
                                                        line=dict(width=1, color=colors_dict[fish_id]),
                                                        track_label=f"culled, Fish {fish_id}")
            particle_plotter_dict[txt_file][fish_id].plot_tracks(RTS_track, mapping_2D,mode="lines",
                                                        uncertainty=uncertainty2D,particle=particle2D,plot_particle_paths=plot_particle_paths2D, 
                                                        line=dict(width=1, color=colors_dict[fish_id]),
                                                        track_label=f"RTS RW, Fish {fish_id}")
        particle_plotter_dict_2D[txt_file][fish_id].fig.update_layout( 
            plot_bgcolor="white",  # Set background color to white
            xaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text="Time", font=dict(size=20)),  # Add large fish_id
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text=fish_id, font=dict(size=20)),  # Add large fish_id
            ),
            legend=dict(
                font=dict(size=15),       # Make the legend font larger
                # orientation='v',
                # xanchor="auto",         # Center the legend
                # yanchor="auto",           # Align the legend to the bottom of the plot
                bordercolor="Black",
                borderwidth=3,
                # y=+0.45,                   # Position it above the graph
                # x=0.6                    # Center it horizontally
            ),
        )
        if save_plots2D:
            particle_plotter_dict_2D[txt_file][fish_id].fig.write_html(str(file_path))
        if show_plots2D:
            particle_plotter_dict_2D[txt_file][fish_id].fig.show()

In [ ]:
particle_plotter_dict_3D ={}
for txt_file, file_data in trajectory_data.items():
    particle_plotter_dict_3D[txt_file]={}
    for fish_id, fish_dict in file_data.items():
        particle_plotter_dict_3D[txt_file][fish_id]=Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.TWO)
        file_path = Path(folder_path + rf"\3D{txt_file}\Fish{fish_id}.html")
        file_path.parent.mkdir(parents=True, exist_ok=True)
        particle_plotter_dict_3D[txt_file][fish_id].plot_ground_truths(trajectory_data[txt_file][fish_id]['plottable_tracks'], [0,1,2], 
                                                            truths_label=f"Fish {fish_id} gt")
        particle_plotter_dict_3D[txt_file][fish_id].plot_tracks(Optimal_lp_tracks[txt_file][fish_id], [0,1,2],
                                                        mode="lines",line=dict(width=1, color=colors_dict[fish_id]),
                                                        track_label=f"Levy RW, Fish {fish_id}")
        
        particle_plotter_dict_3D[txt_file][fish_id].plot_tracks(Optimal_gp_tracks[txt_file][fish_id], [0,1,2],
                                                        mode="lines", line=dict(width=1, color=colors_dict[fish_id], dash='dot'),
                                                        track_label=f"Gaussian RW, Fish {fish_id}")
    particle_plotter_dict_3D[txt_file][fish_id].fig.update_layout( 
        plot_bgcolor="white",  # Set background color to white
        xaxis=dict(
            showgrid=True,
            gridcolor="gray",      # Keep the grid
            title=dict(text="Time", font=dict(size=20)),  # Add large fish_id
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="gray",      # Keep the grid
            title=dict(text=fish_id, font=dict(size=20)),  # Add large fish_id
        ),
        legend=dict(
            font=dict(size=15),       # Make the legend font larger
            # orientation='v',
            # xanchor="auto",         # Center the legend
            # yanchor="auto",           # Align the legend to the bottom of the plot
            bordercolor="Black",
            borderwidth=3,
            # y=+0.45,                   # Position it above the graph
            # x=0.6                    # Center it horizontally
        ),
    )
    if save_plots3D:
        particle_plotter_dict_3D[txt_file][fish_id].fig.write_html(str(file_path))
    if show_plots3D:
        particle_plotter_dict_3D[txt_file][fish_id].fig.show()